# 8 SQL Interview Questions — Executable Databricks Notebook

**Run from top to bottom.**

Each question contains executable:[](url)
1. Databricks SQL
2. Spark SQL through `spark.sql()`
3. PySpark DataFrame API

### Q1. Second Highest Salary

Write a SQL query to find the **second highest distinct salary** from the Employee table.

---

### Q2. Second Highest Salary Department-wise

Write a SQL query to find the **second highest salary in each department**.

---

### Q3. Find Duplicate Records

Write a SQL query to identify **duplicate employee records** based on employee name, department, and salary.

---

### Q4. Remove Duplicates and Keep Latest Record

Given multiple records for the same employee, write a query to **remove duplicates and keep only the latest record** based on `updated_date`.

---

### Q5. Employees Earning More Than Their Managers

Given an Employee table containing `employee_id`, `employee_name`, `salary`, and `manager_id`, find employees whose **salary is greater than their manager's salary**.

---

### Q6. Latest Order for Each Customer

Given a Customers table and Orders table, find the **latest order placed by each customer**.

---

### Q7. Running Total

Given daily sales data, calculate the **running/cumulative sales total for each day**.

---

### Q8. Customers Who Placed Orders Every Month

Given customer order data, find the customers who **placed at least one order in every month** of the given period.

---

The setup below creates the DataFrames and temporary SQL views required by all examples.

![8 sql questions_1789910134410.jpg](./8 sql questions_1789910134410.jpg "8 sql questions_1789910134410.jpg")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Employee table
employee_data = [
    (1, "John", 10, 10000, None),
    (2, "Alice", 10, 15000, 1),
    (3, "Bob", 10, 12000, 1),
    (4, "Carol", 20, 20000, 2),
    (5, "David", 20, 18000, 2),
    (6, "Eva", 20, 18000, 2)
]

employee_df = spark.createDataFrame(
    employee_data,
    ["emp_id", "emp_name", "dept_id", "salary", "manager_id"]
)

# Orders table
orders_data = [
    (1, 101, "2023-01-01", 250),
    (2, 101, "2023-01-01", 250),
    (3, 101, "2023-01-05", 300),
    (4, 102, "2023-02-10", 400),
    (5, 102, "2023-02-10", 400),
    (6, 102, "2023-02-20", 450),
    (7, 103, "2023-03-10", 150),
    (8, 103, "2023-03-10", 150)
]

orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "order_date", "amount"]
)

# Sales table
sales_data = [
    ("2023-01", 1000),
    ("2023-02", 1500),
    ("2023-03", 2000),
    ("2023-04", 1200)
]

sales_df = spark.createDataFrame(
    sales_data,
    ["month", "sales_amount"]
)

# Create temporary views so the SQL and Spark SQL versions can run.
employee_df.createOrReplaceTempView("employee")
orders_df.createOrReplaceTempView("orders")
sales_df.createOrReplaceTempView("sales")

print("Employee")
employee_df.show()

print("Orders")
orders_df.show()

print("Sales")
sales_df.show()

# 1. Find the 2nd Highest Salary
**Question:** Find the second-highest distinct salary.

Expected answer: **12000**

## A. Databricks SQL

In [0]:
%sql
SELECT MAX(salary) AS second_highest_salary
FROM employee
WHERE salary < (SELECT MAX(salary) FROM employee);

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
SELECT MAX(salary) AS second_highest_salary
FROM employee
WHERE salary < (SELECT MAX(salary) FROM employee)
""").show()

## C. PySpark DataFrame API

In [0]:
w = Window.orderBy(F.col("salary").desc())

q1 = (
    employee_df
    .withColumn("rnk", F.dense_rank().over(w))
    .filter(F.col("rnk") == 2)
    .select(F.col("salary").alias("second_highest_salary"))
    .distinct()
)
q1.show()

# 2. Find the 2nd Highest Salary Department-wise
**Question:** Find the second-highest distinct salary for each department.

Expected answer: department 10 → 12000; department 20 → 18000.

## A. Databricks SQL

In [0]:
%sql
WITH ranked AS (
  SELECT dept_id, salary,
         DENSE_RANK() OVER (
            PARTITION BY dept_id ORDER BY salary DESC
         ) AS rnk
  FROM employee
)
SELECT dept_id, salary AS second_highest_salary
FROM ranked
WHERE rnk = 2
ORDER BY dept_id;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
WITH ranked AS (
    SELECT dept_id, salary,
           DENSE_RANK() OVER (
               PARTITION BY dept_id ORDER BY salary DESC
           ) AS rnk
    FROM employee
)
SELECT dept_id, salary AS second_highest_salary
FROM ranked
WHERE rnk = 2
ORDER BY dept_id
""").show()

## C. PySpark DataFrame API

In [0]:
w = Window.partitionBy("dept_id").orderBy(F.col("salary").desc())

q2 = (
    employee_df
    .withColumn("rnk", F.dense_rank().over(w))
    .filter(F.col("rnk") == 2)
    .select("dept_id", F.col("salary").alias("second_highest_salary"))
    .orderBy("dept_id")
)
q2.show()

# 3. Find Duplicate Records
**Question:** Find duplicate records based on `customer_id` and `order_date`.

## A. Databricks SQL

In [0]:
%sql
SELECT customer_id, order_date, COUNT(*) AS duplicate_count
FROM orders
GROUP BY customer_id, order_date
HAVING COUNT(*) > 1
ORDER BY customer_id, order_date;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
SELECT customer_id, order_date, COUNT(*) AS duplicate_count
FROM orders
GROUP BY customer_id, order_date
HAVING COUNT(*) > 1
ORDER BY customer_id, order_date
""").show()

## C. PySpark DataFrame API

In [0]:
q3 = (
    orders_df
    .groupBy("customer_id", "order_date")
    .agg(F.count("*").alias("duplicate_count"))
    .filter(F.col("duplicate_count") > 1)
    .orderBy("customer_id", "order_date")
)
q3.show()

# 4. Remove Duplicates and Keep Only the Latest Record
**Question:** For each customer, keep only the latest order.
If dates tie, use the larger `order_id` as the tie-breaker.

## A. Databricks SQL

In [0]:
%sql
WITH ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY CAST(order_date AS DATE) DESC, order_id DESC
         ) AS rn
  FROM orders
)
SELECT order_id, customer_id, order_date, amount
FROM ranked
WHERE rn = 1
ORDER BY customer_id;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY CAST(order_date AS DATE) DESC, order_id DESC
           ) AS rn
    FROM orders
)
SELECT order_id, customer_id, order_date, amount
FROM ranked
WHERE rn = 1
ORDER BY customer_id
""").show()

## C. PySpark DataFrame API

In [0]:
w = (
    Window.partitionBy("customer_id")
    .orderBy(
        F.to_date("order_date").desc(),
        F.col("order_id").desc()
    )
)

q4 = (
    orders_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("customer_id")
)
q4.show()

# 5. Find Employees Who Earn More Than Their Managers
**Question:** Find employees whose salary is greater than their manager's salary.

## A. Databricks SQL

In [0]:
%sql
SELECT e.emp_id, e.emp_name, e.salary, e.manager_id
FROM employee e
JOIN employee m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
ORDER BY e.emp_id;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
SELECT e.emp_id, e.emp_name, e.salary, e.manager_id
FROM employee e
JOIN employee m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
ORDER BY e.emp_id
""").show()

## C. PySpark DataFrame API

In [0]:
e = employee_df.alias("e")
m = employee_df.alias("m")

q5 = (
    e.join(m, F.col("e.manager_id") == F.col("m.emp_id"), "inner")
    .filter(F.col("e.salary") > F.col("m.salary"))
    .select(
        F.col("e.emp_id"),
        F.col("e.emp_name"),
        F.col("e.salary"),
        F.col("e.manager_id")
    )
    .orderBy("emp_id")
)
q5.show()

# 6. Find the Latest Order for Each Customer
**Question:** Return the complete latest order for every customer.

## A. Databricks SQL

In [0]:
%sql
SELECT order_id, customer_id, order_date, amount
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY CAST(order_date AS DATE) DESC, order_id DESC
         ) AS rn
  FROM orders
)
WHERE rn = 1
ORDER BY customer_id;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
SELECT order_id, customer_id, order_date, amount
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY CAST(order_date AS DATE) DESC, order_id DESC
           ) AS rn
    FROM orders
)
WHERE rn = 1
ORDER BY customer_id
""").show()

## C. PySpark DataFrame API

In [0]:
w = (
    Window.partitionBy("customer_id")
    .orderBy(
        F.to_date("order_date").desc(),
        F.col("order_id").desc()
    )
)

q6 = (
    orders_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("customer_id")
)
q6.show()

# 7. Calculate Running Total of Sales
**Question:** Calculate a running total of sales ordered by month.

Expected totals: **1000, 2500, 4500, 5700**.

## A. Databricks SQL

In [0]:
%sql
SELECT month, sales_amount,
       SUM(sales_amount) OVER (
          ORDER BY month
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM sales
ORDER BY month;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
SELECT month, sales_amount,
       SUM(sales_amount) OVER (
           ORDER BY month
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM sales
ORDER BY month
""").show()

## C. PySpark DataFrame API

In [0]:
w = (
    Window.orderBy("month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q7 = (
    sales_df
    .withColumn("running_total", F.sum("sales_amount").over(w))
    .orderBy("month")
)
q7.show()

# 8. Find Customers Who Placed Orders in Every Month
**Question:** Find customers who placed at least one order in every month of 2023.

The sample data does not contain all 12 months, so this sample returns no rows.
The logic is what you should use in an interview.

## A. Databricks SQL

In [0]:
%sql
WITH monthly_orders AS (
  SELECT DISTINCT customer_id,
         MONTH(CAST(order_date AS DATE)) AS month_num
  FROM orders
  WHERE YEAR(CAST(order_date AS DATE)) = 2023
)
SELECT customer_id
FROM monthly_orders
GROUP BY customer_id
HAVING COUNT(DISTINCT month_num) = 12
ORDER BY customer_id;

## B. Spark SQL (`spark.sql()`)

In [0]:
spark.sql("""
WITH monthly_orders AS (
    SELECT DISTINCT customer_id,
           MONTH(CAST(order_date AS DATE)) AS month_num
    FROM orders
    WHERE YEAR(CAST(order_date AS DATE)) = 2023
)
SELECT customer_id
FROM monthly_orders
GROUP BY customer_id
HAVING COUNT(DISTINCT month_num) = 12
ORDER BY customer_id
""").show()

## C. PySpark DataFrame API

In [0]:
monthly_orders = (
    orders_df
    .filter(F.year(F.to_date("order_date")) == 2023)
    .select(
        "customer_id",
        F.month(F.to_date("order_date")).alias("month_num")
    )
    .distinct()
)

q8 = (
    monthly_orders
    .groupBy("customer_id")
    .agg(F.countDistinct("month_num").alias("month_count"))
    .filter(F.col("month_count") == 12)
    .select("customer_id")
    .orderBy("customer_id")
)
q8.show()

# Interview Quick Revision

- `DENSE_RANK()` → second-highest **distinct** value.
- `ROW_NUMBER()` → keep exactly one row per group.
- `RANK()` → ties share rank, gaps can occur.
- `GROUP BY + HAVING` → find duplicate groups.
- Self join → compare rows in the same table, such as employee vs manager.
- Window `SUM()` → running/cumulative total without collapsing rows.
- `COUNT(DISTINCT month)` → test whether a customer covered all required months.